In [1]:
import os
from tqdm import tqdm
from dotenv import load_dotenv

from llama_index.llms.openai import OpenAI
from llama_index.core.prompts import ChatMessage

from cores.distillation.generic_generation import generic_generate
from cores.utils import filter_query


load_dotenv()
llm = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    model="gpt-4o-mini", 
    temperature=1.0
) # set temp = 1.0


CATEGORY_EXAMPLE_MAPPING = {
    "Đồ uống": {
        "1 tỷ": [
            ["đi ăn nhà hàng sang trọng hết 1 tỏi",
            "mua bánh trung thu hết 2 tỏi rưỡi",
            "trà sữa cho cả công ty mất 3 tỷ 320 triệu",
            "đặt tiệc sinh nhật hết 7 tỷ 500 triệu",
            ],
            ["đi ăn nhà hàng sang trọng hết 1 tỏi mốt",
            "mua trà sữa cho cả công ty mất 2 tỷ mốt",
            "đặt tiệc sinh nhật hết 7 tỷ 5 củ mốt",
            ]
        ]
    },
    "Commute": {
    "1 tỷ": [
        [
            "đổ xăng cả năm hết 1 tỏi",
            "phí gửi xe chung cư hết 2 tỏi rưỡi",
            "bảo hiểm ô tô năm nay mất 3 tỷ 320 triệu",
            "thuê xe chạy hợp đồng hết 7 tỷ 500 triệu"
        ],
        [
            "bảo hiểm xe hơi mất 1 tỏi mốt",
            "đổ xăng đường dài hết 2 tỷ mốt",
            "sửa chữa xe hơi hết 7 tỷ 5 củ mốt",
        ]
    ]
    },
        "Health_Care": {
        "1 tỷ": [
            [
                "phí khám sức khỏe tổng quát hết 1 tỏi",
                "mua thuốc bổ cho cả nhà hết 2 tỏi rưỡi",
                "đăng ký gói tập gym cao cấp mất 3 tỏi 320 triệu",
                "bảo hiểm y tế gia đình trọn gói hết 7 500 triệu"
            ],
            [
                "phí phẫu thuật hết 1 tỏi 4 củ 10 cành",
                "mua thực phẩm chức năng hết 2 tỷ 3 triệu 5 trăm",
                "đăng ký gói tập yoga 5 tỷ 2 củ 8 chục nghìn",
                "bảo hiểm sức khỏe toàn diện hết 3 tỏi 6 triệu 4 loét"
            ]
        ]
    },
    "Living_Expense": {
        "1 tỷ": [
            [
                "tiền điện tháng này hết 1 tỏi",
                "hóa đơn tiền nước cả năm hết 2 tỏi 320 triệu",
                "đóng tiền internet tốc độ cao mất 3 tỏi 2",
                "mua gas dự trữ cho cả năm hết 7 tỏi 500 triệu"
            ],
            [
                "phí truyền hình cáp hết 1 tỏi 4 củ 10 cành",
                "tiền điện thoại di động hết 2 tỏi 3 triệu 5 trăm",
                "tiền đi siêu thị tháng này hết 5 tỏi 2 củ 8 chục nghìn",
                "hóa đơn internet tốc độ cao hết 3 tỏi 6 triệu 4 loét"
            ]
        ]
    }
}

SYSTEM_MSG = ("You are a money manager assistant.\n"
                 "These under examples are sentences about spending money of value {value} VND for {subcategory}\n"
                 "Please generate 5 sentences that have value of {value} VND for {subcategory} similar to the examples.\n"
                 "EXAMPLES:\n{example}")
USER_MSG = "Similar sentences:\n"

SYSTEM_PROMPT = ChatMessage(
    role="system",
    content=SYSTEM_MSG
)
USER_PROMPT = ChatMessage(
    role="user",
    content=USER_MSG
)

for subcategory, value in CATEGORY_EXAMPLE_MAPPING.items():
    for money_value, samples in value.items():
        for i, sample in tqdm(enumerate(samples), desc=money_value):
            if sample: # tránh list rỗng
                example = "\n".join(sample)
                generated_sentences = ""
                for _ in range(5): # 50 câu
                    responses = generic_generate(
                        llm,
                        SYSTEM_PROMPT,
                        USER_PROMPT,
                        prompt_kwargs=dict(
                            value=money_value,
                            subcategory=subcategory,
                            example=example
                        )
                    )
                    responses = responses.split('\n')
                    for raw_response in responses:
                        if raw_response:
                            response = filter_query(raw_response)
                            generated_sentences += f"{response}\n"

                with open(f"data/generated/test_{subcategory}_{money_value}_{i}.txt", 'w', encoding="utf-8") as f:
                    f.write(generated_sentences)

1 tỷ: 2it [00:22, 11.46s/it]
1 tỷ: 2it [00:19,  9.91s/it]
1 tỷ: 2it [00:19,  9.87s/it]
1 tỷ: 2it [00:24, 12.27s/it]


In [2]:
import os

# Đường dẫn tới thư mục chứa các file
folder_path = "data/generated"

# Lấy danh sách file .txt trong thư mục, đảm bảo sắp xếp theo tên để tránh xáo trộn thứ tự
files = sorted([f for f in os.listdir(folder_path) if f.endswith(".txt")])

# Ghép nội dung từ tất cả các file, loại bỏ khoảng trắng dư thừa
all_content = []

for file in files:
    file_path = os.path.join(folder_path, file)
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read().strip()  # Loại bỏ khoảng trắng đầu/cuối của từng file
        if content:  # Chỉ thêm nếu file có nội dung
            all_content.append(content)

# Ghi toàn bộ nội dung vào file mới, mỗi file cách nhau bởi dấu xuống dòng
output_file = "data/generated/merged.txt"
with open(output_file, "w", encoding="utf-8") as f:
    f.write("\n".join(all_content))  # Ghép nội dung với xuống dòng giữa các file

print(f"Đã nối toàn bộ file vào {output_file} mà không có khoảng trắng dư thừa.")


Đã nối toàn bộ file vào data/generated/merged.txt mà không có khoảng trắng dư thừa.


You're an money manager assistant.
 Your job is to find and convert textual money string into integer money string

In [9]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Sinh dữ liệu dựa trên sample
import random
import os
from openai import OpenAI
from dotenv import load_dotenv

# Load API key
load_dotenv()
llm = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def generate_samples(num_samples=50):
    SYSTEM_PROMPT = """
    You're an AI assistant that generates natural Vietnamese sentences about spending money.
    Use informal terms for money values:
    - Triệu: ['triệu', 'm', 'mê', 'củ', 'chai', 'trai']
    - Trăm nghìn: ['trăm', 'lít', 'loét', 'lốp', 'lip', 'líp', 'list']
    - Chục nghìn: ['chục', 'sịch', 'xị', 'sọi']
    - Nghìn: ['k', 'cành', 'nghìn', 'ngàn']
    Include different spending contexts like shopping, entertainment, travel, etc.
    """
    
    USER_PROMPT = """
    Generate {num_samples} unique Vietnamese sentences where people talk about how much money they spent.
    Each sentence should include at least one informal money unit and should be natural and varied.
    Combining keywords between tiers. 
    Centralize data in the form of hundreds of thousands or less.
    Example outputs:
    - "Đi ăn nhà hàng hết 2 loét 3."
    - "Mua đôi giày mới tốn 7 lít mốt."
    - "Mua bánh trung thu hết 2 trăm rưỡi."
    - "Đầu tư bitcoin hết 1 lốp 8 sọi"
    - "Tiền điện tháng này là 2 líp 5 cành"
    """.replace("{num_samples}", str(num_samples))
    
    response = llm.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": USER_PROMPT}
        ],
        temperature=1.0
    ).choices[0].message.content
    
    return response.split("\n")

# Generate data
samples = generate_samples(50)

# Save to file
output_file = "data/generated/generated.txt"
os.makedirs(os.path.dirname(output_file), exist_ok=True)
with open(output_file, "w", encoding="utf-8") as f:
    for sample in samples:
        f.write(sample.strip() + "\n")

print(f"Generated {len(samples)} samples and saved to {output_file}")

Generated 50 samples and saved to data/generated/generated.txt


In [7]:
import os
from tqdm import tqdm
from dotenv import load_dotenv

# from llama_index.llms.openai import OpenAI
from openai import OpenAI
from llama_index.core.prompts import ChatMessage

from cores.distillation.generic_generation import generic_generate
from cores.utils import filter_query


load_dotenv()
llm = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
) # set temp = 1.0

# Prompt to predict the money value of each sentence in file in data/generated/merged.txt
SYSTEM_PROMPT = """
You're an money manager assistant.
Your job is to find and convert textual money string into integer money string
Note that:
- The keywords ["triệu", 'm', "mê", "củ", "chai", "trai"] represent money with value of million
- The keywords ["trăm", "lít", "loét", "lốp", "lip", "líp", "list"] represent money with value of hundred thousand
- The keywords ["chục", "sịch", "xị", "sọi"] represent money with value of ten thousand
- The keywords ["k", "cành", "nghìn", "ngàn"] represent money with value of thousand
- The keywords ["tỷ", "tỉ", "tỏi"] represent money with value of billion
"""

USER_PROMPT = """
Only output the money value in integer, no other text.
Example 1: Input: "Phí bảo trì xe hơi năm nay mất 3 tỏi."	Output: 3000000000
Example 2:
Input: "Chi phí sửa nhà năm nay tốn 2 tỷ 8."
Output: 2800000000
Example 3: 
Input: "Tiền mua xe hơi hạng sang hết 5 tỏi 2 triệu 8 trăm."
Output: 5000200800
Example: 4
Input: "Mua bộ trang sức kim cương hết 7 tỉ 4 triệu 6 trăm."
Output: 7000400600
Example: 5
Input: "Tiền học phí du học năm nay là 1 tỏi 9 triệu 2 trăm."
Output: 1000900200

Real Input:
Input: {input}
"""

SYSTEM_PROMPT_XLSX = """
You're an money manager assistant.
Your job is to find and convert textual money string into integer money string
"""

# Read the file data/generated/merged.txt
with open("data\generated\merged.txt", "r", encoding="utf-8") as f:
    input_data = f.read()

# Predict the money value of each sentence in the file
responses = []
for sentence in input_data.split("\n"):
    value = llm.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": USER_PROMPT.replace("{input}", sentence)}
        ], 
        temperature=1.0
    ).choices[0].message.content
    responses.append({"system": SYSTEM_PROMPT_XLSX, "user": sentence, "value": value})


# Save to xlsx file with system columns is "You're an money manager assistant.Your job is to find and convert textual money string into integer money string", User is text that predict and value is output
import pandas as pd

df = pd.DataFrame(responses)
df.to_excel("bil_data.xlsx", index=False)


In [4]:
import os
import pandas as pd
from tqdm import tqdm
from openai import OpenAI
from dotenv import load_dotenv

# Load API key
load_dotenv()
llm = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))  # set temp = 1.0

# Đọc file Excel
file_path = "bil_data.xlsx"
df = pd.read_excel(file_path)

# Xác định index bắt đầu từ dòng thứ 179 (tức là index 178 trong DataFrame)
start_index = 292

# Prompt
SYSTEM_PROMPT = """
You're a money manager assistant.
Your job is to find and convert textual money string into integer money string
Note that:
- The keywords ["triệu", 'm', "mê", "củ", "chai", "trai"] represent money with value of million
- The keywords ["trăm", "lít", "loét", "lốp", "lip", "líp", "list"] represent money with value of hundred thousand
- The keywords ["chục", "sịch", "xị", "sọi"] represent money with value of ten thousand
- The keywords ["k", "cành", "nghìn", "ngàn"] represent money with value of thousand
- The keywords ["tỷ", "tỉ", "tỏi"] represent money with value of billion
"""

USER_PROMPT = """
Only output the money value in integer, no other text
Nhớ này, với tỏi thì cùng nghĩa là tỷ. Luôn sinh ra đủ 10 chữ số. Có mẫu ở dưới hãy làm theo mẫu đó.
Triệu sẽ nằm ở chữ số thứ 4 trở đi, trăm ngay sau. 
Example 1: Input: "Phí bảo trì xe hơi năm nay mất 3 tỏi."	Output: 3000000000
Example 2:
Input: "Chi phí sửa nhà năm nay tốn 2 tỷ 3 triệu 5 trăm."
Output: 2003500000
Example 3: 
Input: "Tiền mua xe hơi hạng sang hết 5 tỏi 2 triệu 8 trăm."
Output: 5002800000
Example: 4
Input: "Mua bộ trang sức kim cương hết 7 tỉ 4 triệu 6 trăm."
Output: 7004600000
Example: 5
Input: "Tiền học phí du học năm nay là 1 tỏi 9 triệu 2 trăm."
Output: 1009200000

Real Input:
Input: {input}
"""

# Chạy lại dự đoán từ dòng thứ 179 trở đi
for idx in tqdm(range(start_index, len(df)), desc="Processing rows"):
    sentence = df.loc[idx, "user"]
    if pd.notna(sentence):  # Bỏ qua nếu user text bị NaN
        value = llm.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": USER_PROMPT.replace("{input}", sentence)}
            ],
            temperature=1.0
        ).choices[0].message.content
        df.loc[idx, "value"] = value  # Cập nhật giá trị mới

# Ghi lại file Excel
df.to_excel(file_path, index=False)
print(f"Updated {len(df) - start_index} rows in {file_path}")


FileNotFoundError: [Errno 2] No such file or directory: 'bil_data.xlsx'